In [5]:
# ============================================================
# PROJECT 7 EXTENDED: SEC EDGAR FUNDAMENTAL ANALYSIS & VALUATION
# ============================================================
import requests
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. Helper: Get CIK from Ticker using SEC's official mapping
# ------------------------------------------------------------
_cik_mapping = None   # cache for the mapping

def get_cik_from_ticker(ticker):
    """
    Fetch CIK for a given ticker from SEC's company_tickers.json.
    Returns CIK as a 10-digit string with leading zeros.
    """
    global _cik_mapping
    if _cik_mapping is None:
        url = "https://www.sec.gov/files/company_tickers.json"
        headers = {'User-Agent': 'Your Name (your.email@example.com)'}
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            data = response.json()
            # The JSON is a dict with numeric keys; each value contains 'cik_str', 'ticker', 'title'
            _cik_mapping = {}
            for item in data.values():
                cik = str(item['cik_str']).zfill(10)
                ticker_symbol = item['ticker']
                _cik_mapping[ticker_symbol.upper()] = cik
        except Exception as e:
            raise Exception(f"Failed to fetch CIK mapping: {e}")
    ticker_upper = ticker.upper()
    if ticker_upper not in _cik_mapping:
        raise ValueError(f"Ticker {ticker} not found in SEC mapping.")
    return _cik_mapping[ticker_upper]

# ------------------------------------------------------------
# 2. Fetch Financial Data from SEC EDGAR API
# ------------------------------------------------------------
def fetch_company_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {'User-Agent': 'Your Name (your.email@example.com)'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code}")
    return response.json()

# ------------------------------------------------------------
# 3. Extract a Specific Fact for a Given Year (supports multiple fact names)
# ------------------------------------------------------------
def get_fact_value(facts, fact_names, unit='USD', filing_type='10-K', year=None):
    """
    Try multiple fact names (list) and return the first found.
    Returns (value, date) or (None, None).
    """
    if isinstance(fact_names, str):
        fact_names = [fact_names]
    for fname in fact_names:
        if fname in facts['facts']['us-gaap']:
            # Check if the unit exists (some facts may have 'shares' unit)
            if unit in facts['facts']['us-gaap'][fname]['units']:
                data = facts['facts']['us-gaap'][fname]['units'][unit]
            else:
                # If unit not found, try any unit (for shares, maybe 'shares' not present)
                data = []
                for u in facts['facts']['us-gaap'][fname]['units']:
                    data.extend(facts['facts']['us-gaap'][fname]['units'][u])
            # Filter by filing type
            annual = [d for d in data if d['form'] == filing_type]
            if not annual:
                continue
            annual.sort(key=lambda x: x['end'], reverse=True)
            if year:
                for d in annual:
                    if d['end'].startswith(str(year)):
                        return d['val'], d['end']
            else:
                return annual[0]['val'], annual[0]['end']
    return None, None

# ------------------------------------------------------------
# 4. Fetch Multiple Years of Financial Data
# ------------------------------------------------------------
def get_financial_series(facts, fact_name, unit='USD', filing_type='10-K', years=5):
    """
    Retrieve a list of values for the last `years` annual filings.
    Returns a list of tuples (year, value).
    """
    if fact_name not in facts['facts']['us-gaap']:
        return []
    if unit in facts['facts']['us-gaap'][fact_name]['units']:
        data = facts['facts']['us-gaap'][fact_name]['units'][unit]
    else:
        data = []
        for u in facts['facts']['us-gaap'][fact_name]['units']:
            data.extend(facts['facts']['us-gaap'][fact_name]['units'][u])
    annual = [d for d in data if d['form'] == filing_type]
    annual.sort(key=lambda x: x['end'], reverse=True)
    result = []
    for d in annual[:years]:
        year = int(d['end'][:4])
        result.append((year, d['val']))
    return result

# ------------------------------------------------------------
# 5. Compute Key Financial Metrics and Free Cash Flow
# ------------------------------------------------------------
def compute_fcf(ocf, capex):
    return ocf - capex if ocf and capex else None

def get_company_metrics(ticker, facts):
    # Income Statement
    revenue, rev_date = get_fact_value(facts, ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues'])
    net_income, ni_date = get_fact_value(facts, ['NetIncomeLoss', 'NetIncomeLossAvailableToCommonStockholders'])
    # Cash Flow
    ocf, ocf_date = get_fact_value(facts, 'NetCashProvidedByUsedInOperatingActivities')
    capex, capex_date = get_fact_value(facts, 'PaymentsToAcquirePropertyPlantAndEquipment')
    # Balance Sheet
    total_assets, assets_date = get_fact_value(facts, 'Assets')
    total_liabilities, liab_date = get_fact_value(facts, 'Liabilities')
    # Debt
    long_term_debt, ltd_date = get_fact_value(facts, 'LongTermDebt')
    short_term_debt, std_date = get_fact_value(facts, 'ShortTermDebt')
    total_debt = (long_term_debt or 0) + (short_term_debt or 0)
    # Shares Outstanding (multiple possible names)
    shares_names = [
        'WeightedAverageNumberOfSharesOutstandingBasic',
        'WeightedAverageNumberOfDilutedSharesOutstanding',
        'CommonStockSharesOutstanding',
        'EntityCommonStockSharesOutstanding',
        'SharesOutstanding',
        'CommonSharesOutstanding'
    ]
    shares, shares_date = get_fact_value(facts, shares_names, unit='shares')
    if shares is None:
        # Try without specifying unit (may be reported without unit)
        for fname in shares_names:
            if fname in facts['facts']['us-gaap']:
                for u in facts['facts']['us-gaap'][fname]['units']:
                    data = facts['facts']['us-gaap'][fname]['units'][u]
                    annual = [d for d in data if d['form'] == '10-K']
                    if annual:
                        annual.sort(key=lambda x: x['end'], reverse=True)
                        shares = annual[0]['val']
                        shares_date = annual[0]['end']
                        break
                if shares is not None:
                    break

    fcf = compute_fcf(ocf, capex)

    metrics = {
        'ticker': ticker,
        'revenue': revenue,
        'net_income': net_income,
        'ocf': ocf,
        'capex': capex,
        'fcf': fcf,
        'total_assets': total_assets,
        'total_liabilities': total_liabilities,
        'total_debt': total_debt,
        'shares_outstanding': shares,
        'report_date': rev_date
    }
    return metrics

# ------------------------------------------------------------
# 6. Get Historical FCF Series
# ------------------------------------------------------------
def get_fcf_series(facts, years=5):
    ocf_series = get_financial_series(facts, 'NetCashProvidedByUsedInOperatingActivities', years=years)
    capex_series = get_financial_series(facts, 'PaymentsToAcquirePropertyPlantAndEquipment', years=years)
    fcf_by_year = {}
    for year, ocf in ocf_series:
        fcf_by_year[year] = {'ocf': ocf, 'capex': None}
    for year, capex in capex_series:
        if year in fcf_by_year:
            fcf_by_year[year]['capex'] = capex
    result = []
    for year in sorted(fcf_by_year.keys()):
        ocf = fcf_by_year[year]['ocf']
        capex = fcf_by_year[year]['capex']
        if ocf and capex:
            result.append((year, ocf - capex))
    return result

# ------------------------------------------------------------
# 7. Estimate Growth Rate
# ------------------------------------------------------------
def estimate_growth_rate(series):
    if len(series) < 2:
        return 0.03
    first_year, first_val = series[0]
    last_year, last_val = series[-1]
    years = last_year - first_year
    if years <= 0 or first_val <= 0:
        return 0.03
    return (last_val / first_val) ** (1 / years) - 1

# ------------------------------------------------------------
# 8. Approximate WACC
# ------------------------------------------------------------
def estimate_wacc(risk_free_rate=0.03, market_risk_premium=0.05, beta=1.0, tax_rate=0.21,
                  debt_ratio=0.2, cost_of_debt=0.05):
    cost_equity = risk_free_rate + beta * market_risk_premium
    cost_debt_after_tax = cost_of_debt * (1 - tax_rate)
    wacc = (1 - debt_ratio) * cost_equity + debt_ratio * cost_debt_after_tax
    return wacc

# ------------------------------------------------------------
# 9. DCF Valuation
# ------------------------------------------------------------
def dcf_valuation(fcf_last, growth_rate, wacc, terminal_growth=0.02, projection_years=5):
    if fcf_last is None or fcf_last <= 0:
        return None
    projected_fcf = []
    for t in range(1, projection_years+1):
        fcf_t = fcf_last * (1 + growth_rate) ** t
        projected_fcf.append(fcf_t)
    pv_projected = 0
    for t, fcf in enumerate(projected_fcf, 1):
        pv_projected += fcf / (1 + wacc) ** t
    terminal_fcf = projected_fcf[-1] * (1 + terminal_growth)
    terminal_value = terminal_fcf / (wacc - terminal_growth)
    pv_terminal = terminal_value / (1 + wacc) ** projection_years
    return pv_projected + pv_terminal

# ------------------------------------------------------------
# 10. Current Price from Yahoo Finance
# ------------------------------------------------------------
def get_current_price(ticker):
    stock = yf.Ticker(ticker)
    hist = stock.history(period='1d')
    if len(hist) == 0:
        raise ValueError(f"No price data found for {ticker}")
    return hist['Close'].iloc[-1]

# ------------------------------------------------------------
# 11. Main Analysis Function
# ------------------------------------------------------------
def analyze_and_value(ticker):
    print(f"\n{'='*60}")
    print(f"ANALYZING {ticker}")
    print('='*60)

    try:
        cik = get_cik_from_ticker(ticker)
        print(f"CIK: {cik}")
    except Exception as e:
        print(f"Error obtaining CIK: {e}")
        return

    try:
        facts = fetch_company_facts(cik)
    except Exception as e:
        print(f"Error fetching company facts: {e}")
        return

    metrics = get_company_metrics(ticker, facts)
    print("\n--- Latest Financials ---")
    for key, val in metrics.items():
        if isinstance(val, (int, float)) and val is not None:
            print(f"{key:20}: {val:,.0f}")
        else:
            print(f"{key:20}: {val}")

    # Check for missing essential data
    if metrics['fcf'] is None:
        print("Error: Free Cash Flow missing. Cannot perform valuation.")
        return

    # Historical FCF for growth
    fcf_series = get_fcf_series(facts, years=5)
    if len(fcf_series) >= 2:
        growth_rate = estimate_growth_rate(fcf_series)
        print(f"\nHistorical FCF growth rate (CAGR): {growth_rate:.2%}")
    else:
        growth_rate = 0.03
        print(f"\nInsufficient FCF history. Using default growth rate: {growth_rate:.2%}")

    wacc = estimate_wacc()
    print(f"Estimated WACC: {wacc:.2%}")

    enterprise_value = dcf_valuation(metrics['fcf'], growth_rate, wacc)
    if enterprise_value is None:
        print("Cannot compute DCF due to negative or missing FCF.")
        return

    # Net debt (simple, ignoring cash)
    net_debt = metrics['total_debt'] if metrics['total_debt'] else 0
    equity_value = enterprise_value - net_debt
    if equity_value <= 0:
        print("Equity value non‑positive. Cannot compute share price.")
        return

    shares = metrics['shares_outstanding']
    if shares is None or shares <= 0:
        print("Error: Could not retrieve shares outstanding. Cannot compute per‑share value.")
        return

    intrinsic_price = equity_value / shares
    print(f"\nIntrinsic value per share: ${intrinsic_price:.2f}")

    try:
        current_price = get_current_price(ticker)
        print(f"Current market price: ${current_price:.2f}")
    except Exception as e:
        print(f"Could not fetch current price: {e}")
        return

    diff = (intrinsic_price - current_price) / current_price
    if diff > 0.20:
        recommendation = "BUY (more than 20% upside)"
    elif diff < -0.20:
        recommendation = "SELL (more than 20% downside)"
    else:
        recommendation = "HOLD (fairly valued within ±20%)"
    print(f"\nUpside / Downside: {diff:.2%}")
    print(f"Recommendation: {recommendation}")

# ------------------------------------------------------------
# 12. Run
# ------------------------------------------------------------
if __name__ == "__main__":
    ticker = input("Enter a stock ticker (e.g., AAPL): ").strip()
    analyze_and_value(ticker)

Enter a stock ticker (e.g., AAPL): AAPL

ANALYZING AAPL
CIK: 0000320193

--- Latest Financials ---
ticker              : AAPL
revenue             : 416,161,000,000
net_income          : 112,010,000,000
ocf                 : 111,482,000,000
capex               : 12,715,000,000
fcf                 : 98,767,000,000
total_assets        : 359,241,000,000
total_liabilities   : 285,508,000,000
total_debt          : 90,678,000,000
shares_outstanding  : 14,948,500,000
report_date         : 2025-09-27

Historical FCF growth rate (CAGR): -0.41%
Estimated WACC: 7.19%

Intrinsic value per share: $110.47
Current market price: $247.99

Upside / Downside: -55.46%
Recommendation: SELL (more than 20% downside)
